# 01 · Diagnose the rambling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CSroseX/Legal-Llama/blob/main/notebooks/01_diagnose_rambling.ipynb)

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Paste the output of the last cell back to Claude.

In [ ]:
!pip install -q -U bitsandbytes accelerate

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
report = {}

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

DATA_REPO = "theatticusproject/cuad-qa"
DATA_REV = "refs/convert/parquet"
data_files = list_repo_files(DATA_REPO, repo_type="dataset", revision=DATA_REV)

def load_split(split):
    paths = [
        hf_hub_download(DATA_REPO, f, repo_type="dataset", revision=DATA_REV)
        for f in data_files
        if f.startswith(f"default/{split}/") and f.endswith(".parquet")
    ]
    df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
    df["answer_texts"] = df["answers"].apply(lambda a: list(a["text"]))
    df["answer_starts"] = df["answers"].apply(lambda a: [int(s) for s in a["answer_start"]])
    df["has_answer"] = df["answer_texts"].apply(len) > 0
    return df.drop(columns=["answers"])

train = load_split("train")
test = load_split("test")
report["rows"] = {"train": len(train), "test": len(test)}
report["share_with_answer"] = {"train": float(round(train.has_answer.mean(), 3)), "test": float(round(test.has_answer.mean(), 3))}
print(report["rows"], report["share_with_answer"])

In [ ]:
from bisect import bisect_right
from transformers import AutoTokenizer

MODEL = "CSroseX/Legal_llama3.1_part5"
tok = AutoTokenizer.from_pretrained(MODEL)

def analyse(df):
    contracts = df.drop_duplicates("title")[["title", "context"]]
    starts_by_title, length_by_title = {}, {}
    for title, context in zip(contracts.title, contracts.context):
        enc = tok(context, add_special_tokens=False, return_offsets_mapping=True)
        starts_by_title[title] = [s for s, _ in enc["offset_mapping"]]
        length_by_title[title] = len(enc["input_ids"])
    lengths = pd.Series(length_by_title)
    q_tokens = df.question.map(lambda q: len(tok(q, add_special_tokens=False)["input_ids"]))
    a_tokens = df.answer_texts.map(lambda a: len(tok(a[0], add_special_tokens=False)["input_ids"]) if a else 0)
    full = df.title.map(length_by_title) + q_tokens + a_tokens + 60
    answered = df[df.has_answer]
    answer_pos = [
        bisect_right(starts_by_title[t], s[0]) - 1
        for t, s in zip(answered.title, answered.answer_starts)
    ]
    answer_pos = pd.Series(answer_pos)
    return {
        "contracts": int(len(lengths)),
        "contract_tokens_median": int(lengths.median()),
        "contract_tokens_mean": int(lengths.mean()),
        "contract_tokens_max": int(lengths.max()),
        "examples_longer_than_2048": float(round((full > 2048).mean(), 3)),
        "examples_longer_than_8192": float(round((full > 8192).mean(), 3)),
        "answers_starting_after_token_2048": float(round((answer_pos > 2048).mean(), 3)),
        "answers_starting_after_token_8192": float(round((answer_pos > 8192).mean(), 3)),
    }

report["lengths_train"] = analyse(train)
report["lengths_test"] = analyse(test)
print(report["lengths_train"])
print(report["lengths_test"])

In [ ]:
test_lengths = {t: len(tok(c, add_special_tokens=False)["input_ids"]) for t, c in test.drop_duplicates("title")[["title", "context"]].values}

def pick_examples():
    for title in sorted(test_lengths, key=test_lengths.get):
        rows = test[test.title == title]
        yes, no = rows[rows.has_answer], rows[~rows.has_answer]
        if len(yes) >= 2 and len(no) >= 1:
            return pd.concat([yes.iloc[:2], no.iloc[:1]]), title
    raise RuntimeError("no suitable contract")

examples, chosen_title = pick_examples()
report["probe_contract"] = {"title": chosen_title, "tokens": test_lengths[chosen_title]}

import gc
del train
gc.collect()
print(report["probe_contract"])
for q in examples.question:
    print("-", q[:140])

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.eval()
print(f"{torch.cuda.memory_allocated() / 1e9:.1f} GB on GPU")

In [ ]:
TEMPLATES = {
    "plain": "{context}\n\n{question}\n",
    "alpaca": (
        "Below is an instruction that describes a task, paired with an input that provides further context. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{question}\n\n### Input:\n{context}\n\n### Response:\n"
    ),
    "qa": "Context: {context}\n\nQuestion: {question}\n\nAnswer: ",
}

EOS_ID = tok.eos_token_id
STOP_IDS = [EOS_ID, tok.convert_tokens_to_ids("<|eot_id|>")]

def generate(prompt, max_new_tokens=200):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            eos_token_id=STOP_IDS,
            pad_token_id=tok.pad_token_id,
        )
    new = out[0, inputs["input_ids"].shape[1]:]
    stopped = len(new) > 0 and new[-1].item() in STOP_IDS
    return tok.decode(new, skip_special_tokens=True), bool(stopped), int(len(new))

runs = []
for i, row in enumerate(examples.itertuples()):
    expected = row.answer_texts[0] if row.answer_texts else "(no answer in contract)"
    for name, template in TEMPLATES.items():
        text, stopped, n = generate(template.format(context=row.context, question=row.question))
        runs.append({"example": i, "template": name, "stopped_by_eos": stopped, "new_tokens": n,
                     "expected": expected[:200], "output": text[:400]})
        print(f"\n=== example {i} | {name} | stopped={stopped} | tokens={n}")
        print("EXPECTED:", expected[:200])
        print("GOT     :", text[:400])

In [ ]:
def p_eos_after(prompt, answer):
    inputs = tok(prompt + answer, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1].float()
    return round(torch.softmax(logits, -1)[EOS_ID].item(), 4)

eos_probs = []
for i, row in enumerate(examples.itertuples()):
    if not row.answer_texts:
        continue
    for name, template in TEMPLATES.items():
        prompt = template.format(context=row.context, question=row.question)
        eos_probs.append({"example": i, "template": name, "p_eos_after_correct_answer": p_eos_after(prompt, row.answer_texts[0])})
print(pd.DataFrame(eos_probs).to_string(index=False))

In [ ]:
import json
report["generation"] = [{k: r[k] for k in ("example", "template", "stopped_by_eos", "new_tokens")} for r in runs]
report["outputs"] = [{k: r[k][:200] if isinstance(r[k], str) else r[k] for k in ("example", "template", "expected", "output")} for r in runs]
report["eos_probability"] = eos_probs
print(json.dumps(report, indent=1, ensure_ascii=False))